# RAG Ingestion Pipeline (LangChain + Chroma)

This notebook builds the initial RAG data pipeline:
1. Take input text
2. Split text into chunks
3. Create embeddings
4. Store embeddings in a persistent vector database
5. Run cosine-similarity retrieval check

In [ ]:
# Install required packages (run once)
%pip install -qU langchain langchain-community langchain-huggingface langchain-text-splitters langgraph chromadb sentence-transformers


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#imports
from pathlib import Path
from typing import TypedDict
import uuid

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langgraph.graph import END, StateGraph

# path
VECTOR_DB_DIR = (Path.cwd().parent / "vector_db").resolve()
VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)

COLLECTION_NAME = "rag_docs"
# print(f"Vector DB path: {VECTOR_DB_DIR}") 

c:\Users\light\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Vector DB path: C:\Users\light\OneDrive\Desktop\intern\Rag\vector_db


In [ ]:
raw_text = """
Retrieval-Augmented Generation (RAG) improves LLM output by grounding responses in external knowledge.
The pipeline usually has ingestion and retrieval stages.
Ingestion takes source documents, splits them into chunks, embeds them, and stores vectors in a database.
Retrieval finds the most relevant chunks using similarity search, often cosine similarity.
This lets the model answer with context instead of only relying on parametric memory.
""".strip()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=180,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_text(raw_text)
documents = [
    Document(
        page_content=chunk,
        metadata={"source": "manual_input", "chunk_id": idx},
    )
    for idx, chunk in enumerate(chunks)
]
"""
print(f"Total chunks created: {len(documents)}")
for i, d in enumerate(documents[:3]):
    print(f"Chunk {i+1}:", d.page_content)"""

Total chunks created: 3
Chunk 0: Retrieval-Augmented Generation (RAG) improves LLM output by grounding responses in external knowledge.
The pipeline usually has ingestion and retrieval stages.
Chunk 1: Ingestion takes source documents, splits them into chunks, embeds them, and stores vectors in a database.
Chunk 2: Retrieval finds the most relevant chunks using similarity search, often cosine similarity.
This lets the model answer with context instead of only relying on parametric memory.


In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
    persist_directory=str(VECTOR_DB_DIR),
    collection_metadata={"hnsw:space": "cosine"},
)

class IngestState(TypedDict):
    documents: list[Document]
    ids: list[str]
    stored: int


def make_ids(state: IngestState) -> IngestState:
    return {**state, "ids": [str(uuid.uuid4()) for _ in state["documents"]]}


def index_docs(state: IngestState) -> IngestState:
    vectorstore.add_documents(documents=state["documents"], ids=state["ids"])
    return {**state, "stored": len(state["ids"])}


graph = StateGraph(IngestState)
graph.add_node("make_ids", make_ids)
graph.add_node("index_docs", index_docs)
graph.set_entry_point("make_ids")
graph.add_edge("make_ids", "index_docs")
graph.add_edge("index_docs", END)

ingest_app = graph.compile()
result = ingest_app.invoke({"documents": documents, "ids": [], "stored": 0})

print(f"Stored {result['stored']} chunk embeddings in collection '{COLLECTION_NAME}'.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2016.64it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Document(metadata={'source': 'manual_input', 'chunk_id': 0}, page_content='Retrieval-Augmented Generation (RAG) improves LLM output by grounding responses in external knowledge.\nThe pipeline usually has ingestion and retrieval stages.'), Document(metadata={'source': 'manual_input', 'chunk_id': 1}, page_content='Ingestion takes source documents, splits them into chunks, embeds them, and stores vectors in a database.'), Document(metadata={'source': 'manual_input', 'chunk_id': 2}, page_content='Retrieval finds the most relevant chunks using similarity search, often cosine similarity.\nThis lets the model answer with context instead of only relying on parametric memory.')]
Stored 3 chunk embeddings in collection 'rag_docs'.


In [ ]:
query = "How does RAG use chunks and embeddings?"

results = vectorstore.similarity_search_with_score(query, k=3)
for rank, (doc, score) in enumerate(results, start=1):
    print(f"Rank {rank} | Score: {score:.4f}")
    print(doc.page_content)
    print("-" * 80)

Rank 1 | Score: 0.6274
Retrieval-Augmented Generation (RAG) improves LLM output by grounding responses in external knowledge.
The pipeline usually has ingestion and retrieval stages.
--------------------------------------------------------------------------------
Rank 2 | Score: 0.6274
Retrieval-Augmented Generation (RAG) improves LLM output by grounding responses in external knowledge.
The pipeline usually has ingestion and retrieval stages.
--------------------------------------------------------------------------------
Rank 3 | Score: 0.6274
Retrieval-Augmented Generation (RAG) improves LLM output by grounding responses in external knowledge.
The pipeline usually has ingestion and retrieval stages.
--------------------------------------------------------------------------------
